# 文本嵌入模型
Embeddings 类是一个用于与文本嵌入模型接口的类。有很多嵌入大模型供应商（OpenAI、Cohere、Hugging Face 等） - 这个类旨在为它们提供一个标准接口。

## embed

- 将这两个方法分开是因为某些嵌入大模型供应商对文档（待搜索的内容）和查询（搜索查询本身）有不同的嵌入方法。     
    - `.embed_documents`，接受多个文本作为输入
    - `.embed_query`，接受单个文本。
- 返回值：
    - `.embed_query` 将返回一个浮点数列表
    - `.embed_documents` 返回一个浮点数列表的列表。

In [18]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embeddings = HuggingFaceBgeEmbeddings(model_name="moka-ai/m3e-base")

### 使用 .embed_documents 嵌入字符串列表，返回嵌入列表

In [2]:


embed_vectors = embeddings.embed_documents(
    [
        '''登仙台高悬于九天之上，下临无地，四周是翻滚不休、色呈七彩的茫茫云海。''',
        '''云海之中，时有玉光流转，仙影缥缈，''',
        '''那是早已位列仙班的各方仙君，或驾灵兽，或乘宝辇，或静立祥云之端，''',
        '''目光尽数投向那登仙台的中心。''',
    ]
)
len(embed_vectors), len(embed_vectors[0]), type(embed_vectors)

(4, 768, list)

In [3]:
embed_vector = embeddings.embed_query("你好！")
len(embed_vector), type(embed_vector), embed_vector[-5:]

(768,
 list,
 [-1.4307976961135864,
  -0.5786888003349304,
  -0.8872321844100952,
  -0.4537815749645233,
  -0.9592691659927368])

## cache
缓存嵌入可以使用 CacheBackedEmbeddings 来完成。缓存支持的嵌入器是一个包装器，它在一个键值存储中缓存嵌入。 文本被哈希处理，哈希值用作缓存中的键。

- 初始化 CacheBackedEmbeddings 的主要支持方式是 `from_bytes_store`。它接受以下参数：

    - *underlying_embedder*: 用于嵌入的嵌入器。

    - *document_embedding_cache*: 用于缓存文档嵌入的任何 ByteStore。

    - *batch_size*: （可选，默认为 None）在存储更新之间要嵌入的文档数量。

    - *namespace*: （可选，默认为 ``）用于文档缓存的命名空间。此命名空间用于避免与其他缓存的冲突。例如，将其设置为所使用的嵌入模型的名称。

    - *query_embedding_cache*: （可选，默认为 None 或不缓存）用于缓存查询嵌入的  ByteStore，或 True 以使用与 document_embedding_cache 相同的存储。

注意!!!!:

- 确保设置 `namespace` 参数，以避免使用不同嵌入模型嵌入相同文本时发生冲突。

- `CacheBackedEmbeddings` 默认不缓存查询嵌入。要启用查询缓存，需要指定 query_embedding_cache。

### 与向量存储一起使用
**本地文件系统存储嵌入**并使用**FAISS向量存储**进行检索的示例。

In [4]:
import os
os.getcwd()

'C:\\Users\\hhm18\\Desktop\\course'

In [5]:
from langchain.embeddings import CacheBackedEmbeddings

In [6]:
embeddings.model_name

'moka-ai/m3e-base'

In [7]:
from langchain.storage import LocalFileStore
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

underlying_embeddings = HuggingFaceBgeEmbeddings(model_name="moka-ai/m3e-base")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings, 
    store, 
    namespace=underlying_embeddings.model_name
)

C:\Users\hhm18\miniconda3\envs\TrainingCamp\lib\site-packages\langchain\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


在嵌入之前缓存是空的：

In [8]:
list(store.yield_keys())

['moka-ai\\m3e-base9fc223db-af5a-5bb7-b661-fb78b6c505c9',
 'moka-ai\\m3e-basea4034390-57f7-523b-bfad-a2e510b13c8d',
 'moka-ai\\m3e-baseff496e24-b511-56ee-ab70-4bbd3b4ad19f']

In [9]:
# 加载文档并分割，将其拆分为块，嵌入每个块并将其加载到向量存储中。
raw_documents = TextLoader("state2/example/chapter1.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
documents = text_splitter.split_documents(raw_documents)

In [10]:
documents

[Document(metadata={'source': 'state2/example/chapter1.txt'}, page_content='## 剑尊跪求别飞升\n>万载苦修，陆青崖终于迎来飞升天劫。  \n>第九道雷劫劈下时，他却突然张开双臂迎向雷光。  \n>在漫天仙君惊愕的注视中笑道：“这一万年……我装够了。”  \n>“其实我压根不想修仙，只想回村种田。”  \n>正当他准备自碎元神时，天道降下一卷金旨——  \n>上面竟是他三百年前写给自己的婚书。  \n>新娘的名字，是当年为他而死的凡人女子。\n---\n\n万载岁月，弹指一瞬。\n\n登仙台高悬于九天之上，下临无地，四周是翻滚不休、色呈七彩的茫茫云海。云海之中，时有玉光流转，仙影缥缈，那是早已位列仙班的各方仙君，或驾灵兽，或乘宝辇，或静立祥云之端，目光尽数投向那登仙台的中心。\n\n今日，是万载以来第一位修士叩问天门，举霞飞升之期。\n\n主角，是陆青崖。\n\n他一身素白道袍，纤尘不染，立于登仙台中央的浑圆玉石之上。玉石温润，内蕴灵光，与脚下整座登仙台，乃至更深处那支撑天地的巍巍昆仑山脉气机相连。狂风卷起他霜白的长发与宽大的袍袖，猎猎作响，却撼不动他身形分毫。他的面容平静，不见万年苦修终得正果的激动，也无面对最后考验的凝重，只有一种近乎漠然的沉寂。\n\n唯有偶尔，他那双深不见底、映照着下方奔流云海与更上方隐现雷光的眼眸深处，会掠过一丝极淡、极难察觉的疲惫。那疲惫并非源于肉身，而是源自魂魄最深处，积攒了万载光阴，重若星尘。\n\n下方观礼的仙君们，神念交织，虽无声响，却自有灵犀交流。\n\n“青崖仙尊道基之浑厚，堪称我辈楷模，这前八重雷劫，竟毫发无伤。”\n\n“不错，心性更是坚毅无双，万载枯寂，守得灵台清明，殊为不易。”\n\n“只待这最后一道‘太乙破虚劫’落下，洗尽凡尘因果，便可铸就无上仙体，位列我等之中了。”\n\n赞誉无声，却沉甸甸地弥漫在云海之间。\n\n陆青崖微微抬首，望向更高处的苍穹。那里，原本七彩祥云汇聚之处，此刻已被无边无际的墨色劫云取代。云层厚重，缓慢旋转，中心是一个深不见底的漩涡，内里紫白色的电蛇疯狂窜动，凝聚着足以毁灭一方小世界的恐怖力量。毁灭的气息铺天盖地，压得下方云海都为之凝滞，连那些仙君的神念交流也悄然安静下来。\n\n第九道天劫，即将降临。\n\n劫云漩涡

创建向量存储：

In [11]:
%%time
db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 0 ns
Wall time: 36.1 ms


如果我们尝试再次创建向量存储，它会快得多，因为不需要重新计算任何嵌入。

In [12]:
%%time
db2 = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 0 ns
Wall time: 5.24 ms


查看一些创建的嵌入：

In [13]:
list(store.yield_keys())[:]

['moka-ai\\m3e-base9fc223db-af5a-5bb7-b661-fb78b6c505c9',
 'moka-ai\\m3e-basea4034390-57f7-523b-bfad-a2e510b13c8d',
 'moka-ai\\m3e-baseff496e24-b511-56ee-ab70-4bbd3b4ad19f']

## change ByteStore
当然还可以构建储存在RAM里的（暂存）文件

In [14]:
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import InMemoryByteStore

store_ram = InMemoryByteStore()

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings, store_ram, namespace=underlying_embeddings.model_name
)

In [20]:
list(store_ram.yield_keys())

[]

In [22]:
store_ram == store

False